# [Introduction to Data Science](http://datascience-intro.github.io/1MS041-2026/)    
## 1MS041, 2026 
&copy;2026 Raazesh Sainudiin, Benny Avelin. [Attribution 4.0 International     (CC BY 4.0)](https://creativecommons.org/licenses/by/4.0/)

# ProbSS 4 — How estimators behave

## What you will do

You will separate the quantity of interest, the rule used to estimate it, and
the number obtained from one sample. Simulation will show how bias, standard
error, and mean squared error describe different parts of estimation error.
You will then build a bounded interval from real course data and check whether
the independence and stability assumptions are believable.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from pathlib import Path

def course_data(filename):
    candidates = (
        Path("data") / filename,
        Path("master/jp/data") / filename,
    )
    for candidate in candidates:
        if candidate.exists():
            return candidate
    raise FileNotFoundError(
        f"Could not find {filename}. Run this notebook from master/jp "
        "or from the repository root."
    )


## 1. See what repeated samples do

Let $X_1,\ldots,X_n$ be IID Bernoulli$(p)$ observations with $p=0.35$. The estimand is $p$. We compare the sample-mean estimator
$\widehat p=\sum_i X_i/n$ with the shrunken estimator
$\widetilde p=(1+\sum_i X_i)/(n+2)$.

An estimator is a rule before data are observed. An estimate is its numerical value on one observed sample.


In [ ]:
rng = np.random.default_rng(2026)
p = 0.35
n = 40
n_repetitions = 20_000

samples = rng.binomial(1, p, size=(n_repetitions, n))
sample_mean = samples.mean(axis=1)
shrunk = (samples.sum(axis=1) + 1) / (n + 2)

def diagnostics(estimates, target):
    bias = estimates.mean() - target
    se = estimates.std(ddof=1)
    mse = np.mean((estimates - target) ** 2)
    return {"bias": bias, "standard error": se, "MSE": mse}

diagnostic_table = pd.DataFrame(
    {
        "sample mean": diagnostics(sample_mean, p),
        "shrunken estimator": diagnostics(shrunk, p),
    }
).T
diagnostic_table


In [ ]:
one_sample = rng.binomial(1, p, size=n)
estimate = one_sample.mean()
print(f"One observed estimate from the sample-mean rule: {estimate:.3f}")

fig, ax = plt.subplots(figsize=(8, 3.5))
bins = np.linspace(0.05, 0.70, 28)
ax.hist(sample_mean, bins=bins, alpha=0.55, density=True, label="sample mean")
ax.hist(shrunk, bins=bins, alpha=0.55, density=True, label="shrunken")
ax.axvline(p, color="black", linestyle="--", label="estimand p")
ax.set(xlabel="estimate", ylabel="density", title="Monte Carlo sampling distributions")
ax.legend()
plt.show()


Bias and variability are different. The identity
$\operatorname{MSE}(T)=\operatorname{Var}(T)+\operatorname{bias}(T)^2$
explains why a slightly biased rule can have lower MSE.


## 2. A yes/no question from lottery data

The local file contains dated Powerball winning-number records. Before loading the results, define $Y=1$ when at least one of the five white-ball numbers is at least 60. Each draw-level observation is therefore in $[0,1]$.

The estimand is the probability of this event for a future draw under a common, independent draw mechanism. The sample mean is an estimator of that probability. Rule changes or temporal dependence would invalidate the common-IID interpretation and must be checked against an authoritative source before publication.


In [ ]:
powerball = pd.read_csv(course_data("NYPowerBall.csv"))
powerball["Draw Date"] = pd.to_datetime(
    powerball["Draw Date"], format="%m/%d/%Y", errors="raise"
)
parsed = powerball["Winning Numbers"].str.split().map(
    lambda values: [int(value) for value in values]
)
assert parsed.map(len).eq(6).all()

white_balls = parsed.map(lambda values: values[:5])
powerball["large_white_ball"] = white_balls.map(
    lambda values: int(any(value >= 60 for value in values))
)

print(powerball[["Draw Date", "Winning Numbers", "large_white_ball"]].head())
print("Number of draws:", len(powerball))


In [ ]:
alpha = 0.05
y = powerball["large_white_ball"].to_numpy(dtype=float)
n_draws = len(y)
estimate = y.mean()
radius = np.sqrt(np.log(2 / alpha) / (2 * n_draws))
ci = (max(0.0, estimate - radius), min(1.0, estimate + radius))

print(f"Observed estimate: {estimate:.3f}")
print(f"Nominal 95% Hoeffding interval under IID draws: {ci}")


## 3. Check the assumptions


In [ ]:
by_year = (
    powerball.assign(year=powerball["Draw Date"].dt.year)
    .groupby("year")["large_white_ball"]
    .agg(["mean", "count"])
)

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(by_year.index, by_year["mean"], marker="o")
ax.axhline(estimate, color="black", linestyle="--", label="all-draw estimate")
ax.set(
    xlabel="draw year",
    ylabel="sample proportion",
    title="A temporal check of the common-distribution assumption",
    ylim=(-0.05, 1.05),
)
ax.legend()
ax.grid(alpha=0.2)
plt.show()

audit = pd.DataFrame(
    {
        "condition": [
            "fixed binary outcome",
            "bounded observations",
            "independent draws",
            "one common probability over time",
            "future target matches the file period",
        ],
        "status": [
            "specified before the calculation",
            "yes: values are 0 or 1",
            "plausible for separate draws but not verified here",
            "must check historical rule changes",
            "not automatic; the local file ends in 2019",
        ],
    }
)
audit


Optional A/B connection: after the two-group result in Chapter 3 has been introduced, the same logic can be applied to a fixed binary outcome in two groups. Random assignment is needed for a causal treatment interpretation; the interval alone does not create causality.


## Recap

Before you finish, make sure you can:

1. For both Monte Carlo rules, report the estimated bias, standard error, and MSE and explain the trade-off.
2. State the lottery estimand in words and distinguish it from the observed estimate.
3. Report the bounded interval together with the assumption check. Do not report it as a guarantee if the common-IID model is not reasonable.
4. Name one additional source needed to determine whether the draw rules were stable.
